In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
from typing import Optional, List, Tuple, Dict


# =========================
# CONFIG
# =========================
BASE_DIR = "al_trajectory_data_all"

# Original datasets (pool) and the column that defines the objective (not used for matching, but useful for sanity print)
ORIGINAL_DATASETS = {
    "matbench_steels (composition)": ("matbench_steels.csv", "yield strength"),
    "P3HT_dataset": ("P3HT_dataset.csv", "Conductivity (measured) (S/cm)"),
    "Perovskite_dataset": ("Perovskite_dataset.csv", "Instability index"),
    "Membrane_dataset": ("Membrane_dataset.csv", "Elastic Modulus_mean"),
}

# Output root folder
OUT_ROOT = "Sampling Performance Check"

# Which LLM files to include per dataset folder
LLM_GLOB = "llm_experiment_*_trajectory_*.csv"

# Exclude files you may have generated later
EXCLUDE_SUFFIXES = (
    "_with_rerank_score.csv",
)

# Remove the LAST point from each trajectory (final best point)
DROP_LAST_POINT = True

# For string-based matching fallback (if Experiment Index missing)
# We'll try these columns in the *original dataset* as keys.
ORIG_KEY_COL_CANDIDATES = [
    "Formatted_parameters",
    "Formatted Parameters",
    "Formatted_Parameters",
    "Experiment Parameters",
    "Parameters",
    "parameter",
    "params",
]

# For string-based matching fallback (if Experiment Index missing)
# We'll try these columns in the *trajectory file* as keys.
TRAJ_KEY_COL_CANDIDATES = [
    "Experiment Parameters",
    "Experiment parameter",
    "Parameters",
    "Selected Parameters",
]

# =========================
# Helpers
# =========================
def _norm(s: str) -> str:
    """Normalize whitespace for stable string matching."""
    return " ".join(str(s).strip().split())

def _pick_col(cols: List[str], candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    return None

def _safe_read_csv(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def _dataset_out_dir(dataset_name: str) -> str:
    # Keep folder name same as dataset folder, but inside OUT_ROOT
    return os.path.join(OUT_ROOT, dataset_name)

def _list_llm_files(dataset_folder: str) -> List[str]:
    files = glob.glob(os.path.join(dataset_folder, LLM_GLOB))
    files = [f for f in files if not any(f.endswith(suf) for suf in EXCLUDE_SUFFIXES)]
    return sorted(set(files))

def _drop_last(df: pd.DataFrame) -> pd.DataFrame:
    if df.shape[0] <= 1:
        return df.copy()
    return df.iloc[:-1].copy()

def _extract_selected_indices_from_trajectory(df_traj: pd.DataFrame) -> Tuple[Optional[np.ndarray], str]:
    """
    Return (indices, method_string).
    Prefer Experiment Index. If missing, return (None, "no_index").
    """
    if "Experiment Index" in df_traj.columns:
        idx = pd.to_numeric(df_traj["Experiment Index"], errors="coerce").dropna().astype(int).to_numpy()
        return idx, "experiment_index"
    return None, "no_index"

def _subset_by_indices(df_orig: pd.DataFrame, idx: np.ndarray) -> Tuple[pd.DataFrame, Dict]:
    """
    Subset original df using integer positions.
    Returns subset df and stats.
    """
    stats = {}
    N = len(df_orig)

    idx = np.asarray(idx, dtype=int)
    stats["n_idx_raw"] = int(idx.size)
    stats["n_idx_unique"] = int(np.unique(idx).size)

    # valid range
    valid_mask = (idx >= 0) & (idx < N)
    idx_valid = idx[valid_mask]
    idx_invalid = idx[~valid_mask]

    stats["n_idx_valid"] = int(idx_valid.size)
    stats["n_idx_invalid"] = int(idx_invalid.size)
    stats["invalid_examples"] = idx_invalid[:10].tolist() if idx_invalid.size else []

    # Subset using iloc (keeps duplicates if present in idx_valid)
    df_sub = df_orig.iloc[idx_valid].copy()
    return df_sub, stats

def _subset_by_string_key(df_orig: pd.DataFrame, df_traj: pd.DataFrame) -> Tuple[pd.DataFrame, Dict, str]:
    """
    Fallback matching:
      - pick a key column from orig
      - pick a key column from traj
      - normalize strings
      - map traj key -> orig row (first occurrence)
    """
    stats = {}

    orig_key_col = _pick_col(list(df_orig.columns), ORIG_KEY_COL_CANDIDATES)
    traj_key_col = _pick_col(list(df_traj.columns), TRAJ_KEY_COL_CANDIDATES)

    if orig_key_col is None or traj_key_col is None:
        stats["error"] = f"Could not find key columns. orig_key_col={orig_key_col}, traj_key_col={traj_key_col}"
        return df_orig.iloc[0:0].copy(), stats, "string_key_missing_cols"

    # build lookup from orig key -> first row index
    orig_keys = df_orig[orig_key_col].astype(str).map(_norm)
    # dropna-ish + duplicates
    # keep first occurrence (consistent with your membrane mapping approach)
    orig_key_to_i = {}
    for i, k in enumerate(orig_keys):
        if k and k not in orig_key_to_i:
            orig_key_to_i[k] = i

    traj_keys = df_traj[traj_key_col].astype(str).map(_norm).to_numpy()
    idx = []
    missing = []
    for k in traj_keys:
        if k in orig_key_to_i:
            idx.append(orig_key_to_i[k])
        else:
            missing.append(k)

    stats["orig_key_col"] = orig_key_col
    stats["traj_key_col"] = traj_key_col
    stats["n_traj_keys"] = int(len(traj_keys))
    stats["n_matched"] = int(len(idx))
    stats["n_missing"] = int(len(missing))
    stats["missing_examples"] = missing[:5]

    df_sub = df_orig.iloc[idx].copy()
    return df_sub, stats, "string_key"

def _prepend_tracking_cols(df_sub: pd.DataFrame, df_traj_used: pd.DataFrame, traj_file: str) -> pd.DataFrame:
    """
    Add helpful tracking columns while keeping all original dataset columns.
    These extra columns are useful later and don't remove any original data.
    """
    out = df_sub.copy()

    # best-effort attach iteration if lengths align
    if "Iteration" in df_traj_used.columns and len(df_traj_used) == len(out):
        out.insert(0, "Iteration", df_traj_used["Iteration"].to_numpy())
    else:
        out.insert(0, "Iteration", np.arange(1, len(out) + 1))

    out.insert(0, "Trajectory_File", os.path.basename(traj_file))
    return out

def _save_subset(df_sub: pd.DataFrame, dataset_name: str, orig_csv_path: str, traj_file: str) -> str:
    out_dir = _dataset_out_dir(dataset_name)
    _ensure_dir(out_dir)

    orig_base = os.path.splitext(os.path.basename(orig_csv_path))[0]
    npts = df_sub.shape[0]
    traj_base = os.path.splitext(os.path.basename(traj_file))[0]

    # user asked: "original dataset name_number of data points"
    # to avoid overwrite across trajectories, append the trajectory basename too
    out_name = f"{orig_base}_{npts}__{traj_base}.csv"
    out_path = os.path.join(out_dir, out_name)
    df_sub.to_csv(out_path, index=False)
    return out_path

def process_one_dataset(dataset_name: str, orig_csv: str, y_col: str) -> None:
    dataset_folder = os.path.join(BASE_DIR, dataset_name)
    if not os.path.isdir(dataset_folder):
        print(f"\n[SKIP] Dataset folder not found: {dataset_folder}")
        return

    if not os.path.exists(orig_csv):
        print(f"\n[SKIP] Original dataset CSV not found: {orig_csv} (needed for {dataset_name})")
        return

    df_orig = _safe_read_csv(orig_csv)
    N = len(df_orig)

    llm_files = _list_llm_files(dataset_folder)
    print(f"\n==============================")
    print(f"Dataset: {dataset_name}")
    print(f"Original: {orig_csv}  (N={N})")
    if y_col in df_orig.columns:
        y = pd.to_numeric(df_orig[y_col], errors="coerce")
        print(f"Objective column '{y_col}': non-null={int(y.notna().sum())}, unique={int(y.dropna().nunique())}")
    else:
        print(f"Objective column '{y_col}': NOT FOUND in original dataset (ok for matching if using indices/keys)")
    print(f"Found LLM trajectory files: {len(llm_files)}")
    print(f"Output folder: {_dataset_out_dir(dataset_name)}")
    print(f"==============================\n")

    if not llm_files:
        return

    # Dataset-level sanity: duplicate objective values (useful if you later do value-based metrics)
    if y_col in df_orig.columns:
        y = pd.to_numeric(df_orig[y_col], errors="coerce").dropna()
        dup = int(len(y) - y.nunique())
        max_rep = int(y.value_counts().max()) if not y.empty else 0
        print(f"[SANITY] Duplicate target entries: {dup} (max repetition of a value: {max_rep})")

    # process each trajectory
    for traj_file in llm_files:
        df_traj = _safe_read_csv(traj_file)

        n_traj_raw = len(df_traj)
        df_traj_used = _drop_last(df_traj) if DROP_LAST_POINT else df_traj.copy()
        n_traj_used = len(df_traj_used)

        # 1) Prefer Experiment Index
        idx, idx_method = _extract_selected_indices_from_trajectory(df_traj_used)

        if idx is not None and idx.size:
            df_sub, st = _subset_by_indices(df_orig, idx)
            match_method = idx_method
            extra_stats = st
            # If we used indices, lengths should match "valid indices count"
            # df_sub keeps duplicates if idx has duplicates.
        else:
            # 2) Fallback: string key matching
            df_sub, st, match_method = _subset_by_string_key(df_orig, df_traj_used)
            extra_stats = st

        # attach tracking cols (keeps all original columns)
        df_sub_out = _prepend_tracking_cols(df_sub, df_traj_used.iloc[:len(df_sub)].copy(), traj_file)

        # save
        out_path = _save_subset(df_sub_out, dataset_name, orig_csv, traj_file)

        # ---- per-file sanity stats ----
        print(f"[FILE] {os.path.basename(traj_file)}")
        print(f"  Trajectory rows: raw={n_traj_raw}, used(after_drop_last={DROP_LAST_POINT})={n_traj_used}")
        print(f"  Match method: {match_method}")
        print(f"  Subset rows saved: {len(df_sub_out)} → {out_path}")

        # index stats if used
        if match_method == "experiment_index":
            print(f"  idx: raw={extra_stats.get('n_idx_raw')}, unique={extra_stats.get('n_idx_unique')}, "
                  f"valid={extra_stats.get('n_idx_valid')}, invalid={extra_stats.get('n_idx_invalid')}")
            if extra_stats.get("n_idx_invalid", 0) > 0:
                print(f"  invalid idx examples: {extra_stats.get('invalid_examples')}")

            # duplicate index warning
            if extra_stats.get("n_idx_unique", 0) < extra_stats.get("n_idx_raw", 0):
                print("  [WARN] Duplicate Experiment Index values found within this trajectory.")
        else:
            # string-match stats
            if "error" in extra_stats:
                print(f"  [WARN] {extra_stats['error']}")
            else:
                print(f"  key cols: traj='{extra_stats.get('traj_key_col')}', orig='{extra_stats.get('orig_key_col')}'")
                print(f"  matched={extra_stats.get('n_matched')}/{extra_stats.get('n_traj_keys')} "
                      f"(missing={extra_stats.get('n_missing')})")
                if extra_stats.get("n_missing", 0) > 0:
                    print(f"  missing examples: {extra_stats.get('missing_examples')}")

        # optional: objective distribution sanity for the subset
        if y_col in df_orig.columns and y_col in df_sub_out.columns:
            yy = pd.to_numeric(df_sub_out[y_col], errors="coerce").dropna()
            if not yy.empty:
                print(f"  subset objective '{y_col}': min={yy.min():.6g}, max={yy.max():.6g}, "
                      f"mean={yy.mean():.6g}, unique={yy.nunique()}")
        print("")

def main():
    _ensure_dir(OUT_ROOT)
    for ds, (orig_csv, y_col) in ORIGINAL_DATASETS.items():
        process_one_dataset(ds, orig_csv, y_col)

if __name__ == "__main__":
    main()


Dataset: matbench_steels (composition)
Original: matbench_steels.csv  (N=312)
Objective column 'yield strength': non-null=312, unique=270
Found LLM trajectory files: 20
Output folder: Sampling Performance Check\matbench_steels (composition)

[SANITY] Duplicate target entries: 42 (max repetition of a value: 4)
[FILE] llm_experiment_yield_strength_trajectory_parameters_seed38.csv
  Trajectory rows: raw=55, used(after_drop_last=True)=54
  Match method: experiment_index
  Subset rows saved: 54 → Sampling Performance Check\matbench_steels (composition)\matbench_steels_54__llm_experiment_yield_strength_trajectory_parameters_seed38.csv
  idx: raw=54, unique=54, valid=54, invalid=0
  subset objective 'yield strength': min=1005.9, max=2356.4, mean=1566.83, unique=49

[FILE] llm_experiment_yield_strength_trajectory_parameters_seed39.csv
  Trajectory rows: raw=3, used(after_drop_last=True)=2
  Match method: experiment_index
  Subset rows saved: 2 → Sampling Performance Check\matbench_steels (com

In [4]:
"""
Warm-start ML Active Learning from LLM-selected subsets + matched random baselines.

Updates in this version:
- Uses YOUR exact feature/target setups per dataset (slicing/dropping as you specified).
- Perovskite is treated as minimization by setting y_eff = -Instability index (exactly like your snippet).
- Adds NaN/inf handling via SimpleImputer(strategy="mean") fit ONLY on the current training set
  (warmstart + sequentially added points), then applied to candidates.
- Windows-safe filenames + short hashed subset keys.
"""

import os
import re
import glob
import random
import hashlib
from pathlib import Path
from typing import Dict, Tuple, Sequence, List

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample
from xgboost import XGBRegressor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions.normal import Normal
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# Config
# ============================================================
DATASET_CSVS = {
    "matbench_steels (composition)": "matbench_steels.csv",
}

SAMPLING_CHECK_ROOT = "Sampling Performance Check"
OUT_BASE = "al_trajectory_data_all"

MODEL_NAMES = ("BNN", "XGB", "RFR", "GPR")
ALPHAS = (0.1, 0.5, 1.0)

SUBSET_FILE_FILTER = re.compile(r".*llm_experiment_.*\.csv$", re.IGNORECASE)


# ============================================================
# Helpers
# ============================================================
_seed_re = re.compile(r"(?i)seed(\d+)")


def parse_seed_from_name(s: str, default: int = 42) -> int:
    m = _seed_re.search(s)
    return int(m.group(1)) if m else default


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_name(s: str, max_len: int = 140) -> str:
    s = re.sub(r'[<>:"/\\|?*\n\r\t]', "_", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    return s[:max_len]


def short_hash(s: str, n: int = 10) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:n]


def alpha_tag(alpha: float) -> str:
    a = float(alpha)
    if abs(a - round(a)) < 1e-12:
        return str(int(round(a)))
    s = f"{a:.6g}"
    return s.replace(".", "p").replace("-", "m")


# ============================================================
# Dataset utilities (YOUR exact setups)
# ============================================================
def is_minimization_dataset(dataset_name: str) -> bool:
    # Perovskite: you defined y_df = -Instability index (maximize -instability == minimize instability)
    return dataset_name == "Perovskite_dataset"


def get_objective_column(dataset_name: str) -> str:
    if dataset_name == "matbench_steels (composition)":
        return "yield strength"
    if dataset_name == "P3HT_dataset":
        return "Conductivity (measured) (S/cm)"
    if dataset_name == "Perovskite_dataset":
        return "Instability index"
    if dataset_name == "Membrane_dataset":
        return "Elastic Modulus_mean"
    raise ValueError(f"Unknown dataset_name: {dataset_name}")


def build_features_and_targets(df: pd.DataFrame, dataset_name: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Returns:
      X_df (features in your defined column selection)
      y_true (original target, in original units/sign)
      y_eff  (effective target used for acquisition; for Perovskite it's negative)
    """
    if dataset_name == "matbench_steels (composition)":
        # Your snippet (note: you *finally* used iloc[:,2:16])
        X_df = df.iloc[:, 2:16].copy()
        y_true = pd.to_numeric(df["yield strength"], errors="coerce").to_numpy(dtype=float)
        y_eff = y_true.copy()
        return X_df, y_true, y_eff

    if dataset_name == "P3HT_dataset":
        X_df = df.iloc[:, 0:5].copy()
        y_true = pd.to_numeric(df["Conductivity (measured) (S/cm)"], errors="coerce").to_numpy(dtype=float)
        y_eff = y_true.copy()
        return X_df, y_true, y_eff

    if dataset_name == "Perovskite_dataset":
        X_df = df.iloc[:, 0:3].copy()
        y_true = pd.to_numeric(df["Instability index"], errors="coerce").to_numpy(dtype=float)
        y_eff = -y_true  # EXACTLY like your snippet
        return X_df, y_true, y_eff

    if dataset_name == "Membrane_dataset":
        X_df = df.drop([
            'Elastic Modulus_mean', 'Elastic Modulus_std', 'Yield Strength_mean',
            'Yield Strength_std', 'Creep Strain_mean', 'Creep Strain_std',
            'Plateau Slope_mean', 'Plateau Slope_std', 'Densification Slope_mean',
            'Densification Slope_std', 'Changepoint_mean', 'Changepoint_std', 'Date',
            'Average Standard Deviation_mean', 'Average Standard Deviation_std',
            'Coefficient of Variation_mean', 'Coefficient of Variation_std',
            'Coefficient of Variation', 'Batch', 'Sample', 'Index',
            'Formatted_Parameters', 'Report', 'Report with output',
            'Formatted_Parameters with output'
        ], axis=1, errors="ignore").copy()

        y_true = pd.to_numeric(df["Elastic Modulus_mean"], errors="coerce").to_numpy(dtype=float)
        y_eff = y_true.copy()
        return X_df, y_true, y_eff

    raise ValueError(f"Unknown dataset_name: {dataset_name}")


def coerce_numeric_X(X_df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure X is numeric; turn non-numeric into NaN so imputer can handle it.
    Also converts inf/-inf to NaN.
    """
    X_num = X_df.apply(pd.to_numeric, errors="coerce")
    X_num = X_num.replace([np.inf, -np.inf], np.nan)
    return X_num


# ============================================================
# Warm-start index recovery (robust)
# ============================================================
def recover_indices_from_subset(
    df_full: pd.DataFrame,
    df_subset: pd.DataFrame,
    *,
    index_col_candidates=("Experiment Index", "Experiment_Index", "Index", "index", "Idx", "ID", "id"),
) -> np.ndarray:
    for c in index_col_candidates:
        if c in df_subset.columns:
            idx = pd.to_numeric(df_subset[c], errors="coerce").dropna().astype(int).to_numpy()
            return idx

    ignore = {
        "Iteration", "Stopping Reason", "Phase",
        "Predicted Target Value", "Uncertainty",
        "Rerank_Score_v35", "Rerank_Score",
    }
    shared = [c for c in df_subset.columns if c in df_full.columns and c not in ignore]
    if not shared:
        raise ValueError("Cannot recover indices: no index column and no shared columns.")

    full_hash = pd.util.hash_pandas_object(df_full[shared], index=False)
    sub_hash = pd.util.hash_pandas_object(df_subset[shared], index=False)

    map_hash_to_indices: Dict[int, List[int]] = {}
    for i, h in enumerate(full_hash.values):
        map_hash_to_indices.setdefault(int(h), []).append(i)

    matched = []
    missing = 0
    for h in sub_hash.values:
        h = int(h)
        if h not in map_hash_to_indices or len(map_hash_to_indices[h]) == 0:
            missing += 1
            continue
        matched.append(map_hash_to_indices[h].pop(0))

    if missing > 0:
        raise ValueError(f"Subset row matching failed: missing {missing}/{len(df_subset)} rows.")
    return np.array(matched, dtype=int)


# ============================================================
# Models (Perovskite+GPR scaling fix kept)
# ============================================================
class GPR:
    def __init__(self, kernel=None, alpha: float = 1.0, random_state: int = 42, y_scale_factor: float = 1.0):
        self.alpha = float(alpha)
        self.y_scale_factor = float(y_scale_factor)
        self.kernel = kernel if kernel else (
            C(1.0, (1e-5, 1e5)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3)) +
            WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-3, 1e6))
        )
        self.model = GaussianProcessRegressor(kernel=self.kernel, n_restarts_optimizer=10, random_state=random_state)

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        self.model.fit(X_train, y_train * self.y_scale_factor)

    def select_next_point(self, X_candidates: np.ndarray):
        mean_s, std_s = self.model.predict(X_candidates, return_std=True)
        mean = mean_s / self.y_scale_factor
        std = std_s / self.y_scale_factor
        ucb = mean + self.alpha * std
        return int(np.argmax(ucb)), ucb, mean, std


class RFR:
    def __init__(self, n_estimators=400, alpha: float = 1.0, random_state: int = 42):
        self.alpha = float(alpha)
        self.model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        self.model.fit(X_train, y_train)

    def rfPredictionIntervals(self, xVal: np.ndarray, percentile: float = 90):
        y_preds = np.stack([t.predict(xVal) for t in self.model.estimators_], axis=1)
        q_down = (100 - percentile) / 2.0
        q_up = 100 - q_down
        y_lower = np.percentile(y_preds, q_down, axis=1)
        y_upper = np.percentile(y_preds, q_up, axis=1)
        y_mean = self.model.predict(xVal)
        y_uncert = y_upper - y_lower
        return y_mean, y_uncert

    def select_next_point(self, X_candidates: np.ndarray):
        mean, uncertainty = self.rfPredictionIntervals(X_candidates)
        ucb = mean + self.alpha * uncertainty
        return int(np.argmax(ucb)), ucb, mean, uncertainty


class XGB:
    def __init__(self, n_estimators=400, n_models=30, alpha: float = 1.0, random_state: int = 42):
        self.alpha = float(alpha)
        self.n_models = int(n_models)
        self.n_estimators = int(n_estimators)
        self.random_state = int(random_state)
        self.models: List[XGBRegressor] = []

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        self.models = []
        rng = np.random.RandomState(self.random_state)
        for _ in range(self.n_models):
            X_s, y_s = resample(X_train, y_train, random_state=rng.randint(0, 10000))
            m = XGBRegressor(
                n_estimators=self.n_estimators,
                reg_alpha=0,
                scale_pos_weight=1,
                base_score=0.5,
                random_state=rng.randint(0, 10000),
            )
            m.fit(X_s, y_s)
            self.models.append(m)

    def select_next_point(self, X_candidates: np.ndarray):
        y_preds = np.column_stack([m.predict(X_candidates) for m in self.models])
        y_mean = y_preds.mean(axis=1)
        y_std = y_preds.std(axis=1)
        ucb = y_mean + self.alpha * y_std
        return int(np.argmax(ucb)), ucb, y_mean, y_std


# ---------------- BNN ----------------
class BayesianLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight_mu = nn.Parameter(torch.Tensor(out_features, in_features).uniform_(-0.2, 0.2))
        self.weight_log_sigma = nn.Parameter(torch.Tensor(out_features, in_features).fill_(-5.0))
        self.bias_mu = nn.Parameter(torch.Tensor(out_features).uniform_(-0.2, 0.2))
        self.bias_log_sigma = nn.Parameter(torch.Tensor(out_features).fill_(-5.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w_sigma = torch.exp(self.weight_log_sigma)
        b_sigma = torch.exp(self.bias_log_sigma)
        w = self.weight_mu + w_sigma * torch.randn_like(self.weight_mu, device=x.device)
        b = self.bias_mu + b_sigma * torch.randn_like(self.bias_mu, device=x.device)
        return F.linear(x, w, b)

    def kl_loss(self) -> torch.Tensor:
        w_sigma = torch.exp(self.weight_log_sigma)
        b_sigma = torch.exp(self.bias_log_sigma)
        w_kl = (torch.log(1.0 / w_sigma) + (w_sigma**2 + self.weight_mu**2 - 1) / 2).sum()
        b_kl = (torch.log(1.0 / b_sigma) + (b_sigma**2 + self.bias_mu**2 - 1) / 2).sum()
        return w_kl + b_kl


class BayesianNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.layers = nn.ModuleList([
            BayesianLinear(input_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
            BayesianLinear(hidden_dim, hidden_dim),
        ])
        self.out = BayesianLinear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.out(x)

    def kl_loss(self) -> torch.Tensor:
        return sum(l.kl_loss() for l in self.layers) + self.out.kl_loss()


def elbo_loss(pred: torch.Tensor, y: torch.Tensor, kl: torch.Tensor, beta: float = 1.0) -> torch.Tensor:
    # pred: (B,1) or (B,) ; y: (B,) or (B,1)
    pred_vec = pred.view(-1)
    y_vec = y.view(-1)
    mse = F.mse_loss(pred_vec, y_vec, reduction="mean")
    return 0.5 * mse + beta * kl


def train_bnn(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    n_epochs: int = 600,
    lr: float = 1e-3,
    beta: float = 1.0,
    batch_size: int = 64,
    device: str = "cpu",
):
    ds = TensorDataset(X, y)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    for _ in range(n_epochs):
        for Xb, yb in dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            p = model(Xb)
            kl = model.kl_loss()
            loss = elbo_loss(p, yb, kl, beta)
            loss.backward()
            opt.step()


def predict_with_uncertainty(model: nn.Module, X: torch.Tensor, n_samples: int = 200):
    """
    Always return mean/std as 1D numpy arrays of length N (N = X.shape[0]).
    Works even when N==1.
    """
    model.eval()
    with torch.no_grad():
        preds = []
        for _ in range(n_samples):
            out = model(X)          # (N,1) typically
            out = out.view(-1)      # -> (N,)
            preds.append(out)
        preds = torch.stack(preds, dim=0)   # (S, N)

        mean = preds.mean(dim=0).cpu().numpy()  # (N,)
        std  = preds.std(dim=0).cpu().numpy()   # (N,)
    return mean, std


class BNN:
    def __init__(self, input_dim: int, device: str = "cpu", alpha: float = 1.0):
        self.alpha = float(alpha)
        self.device = device
        self.model = BayesianNN(input_dim).to(device)

    def update_model(self, X_train: np.ndarray, y_train: np.ndarray):
        Xt = torch.from_numpy(X_train).float().to(self.device)
        yt = torch.from_numpy(y_train).float().to(self.device)
        train_bnn(self.model, Xt, yt, device=self.device)

    def select_next_point(self, X_candidates: np.ndarray):
        Xc = torch.from_numpy(X_candidates).float().to(self.device)
        mean, std = predict_with_uncertainty(self.model, Xc)
        ucb = mean + self.alpha * std
        return int(np.argmax(ucb)), ucb, mean, std


def make_model(model_name: str, *, alpha: float, random_seed: int, input_dim: int, dataset_name: str):
    if model_name == "GPR":
        y_scale = 1e-3 if dataset_name == "Perovskite_dataset" else 1.0
        return GPR(alpha=alpha, random_state=random_seed, y_scale_factor=y_scale)
    if model_name == "RFR":
        return RFR(alpha=alpha, random_state=random_seed)
    if model_name == "XGB":
        return XGB(alpha=alpha, random_state=random_seed)
    if model_name == "BNN":
        device = "cuda" if torch.cuda.is_available() else "cpu"
        return BNN(input_dim, alpha=alpha, device=device)
    raise ValueError("Invalid model name")


# ============================================================
# AL runner (with imputation; min/max handled exactly as desired)
# ============================================================
def run_MLAL_warmstart(
    df_full: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    alpha: float,
    random_seed: int,
    warmstart_indices: np.ndarray,
    *,
    run_kind: str,      # "warm" or "rand"
    subset_key: str,    # short identifier (already sanitized)
    out_dir: str,
) -> Tuple[str, pd.DataFrame]:
    set_all_seeds(random_seed)

    y_col = get_objective_column(dataset_name)

    # Build X/y using YOUR setups
    X_df_raw, y_true, y_eff = build_features_and_targets(df_full, dataset_name)
    X_df = coerce_numeric_X(X_df_raw)

    X = X_df.to_numpy(dtype=float)
    N = X.shape[0]
    d = X.shape[1]

    # Determine "best" in true space for logging / stopping
    minimize = is_minimization_dataset(dataset_name)
    if minimize:
        # y_eff = -y_true, maximizing y_eff => minimizing y_true
        best_true = float(np.nanmin(y_true))
        best_label = f"Min {y_col} in Dataset"
    else:
        best_true = float(np.nanmax(y_true))
        best_label = f"Max {y_col} in Dataset"

    warmstart_indices = np.asarray(warmstart_indices, dtype=int)
    warmstart_indices = warmstart_indices[(warmstart_indices >= 0) & (warmstart_indices < N)]
    warmstart_indices = np.unique(warmstart_indices)
    if warmstart_indices.size == 0:
        raise ValueError("No valid warm-start indices")

    # Initialize training
    selected = warmstart_indices.tolist()
    X_train = X[warmstart_indices]
    y_train_eff = y_eff[warmstart_indices]

    # ---- Fit imputer+scaler on TRAIN ONLY (warmstart), then apply to candidates
    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)

    model = make_model(model_name, alpha=alpha, random_seed=random_seed, input_dim=d, dataset_name=dataset_name)

    # Trajectory rows: log warmstart points as "observed"
    rows = []
    for t, idx in enumerate(warmstart_indices.tolist()):
        rows.append({
            "Iteration": int(t),
            "Index": int(idx),
            "Observed Target Value": float(y_true[idx]),
            "Predicted Target Value": np.nan,
            "Uncertainty": np.nan,
            best_label: float(best_true),
            "Phase": "WarmStart",
            "Stopping Reason": "WarmStart",
        })

    it = warmstart_indices.size
    while True:
        # Train surrogate
        model.update_model(X_train_scaled, y_train_eff)

        # Candidates
        available = np.array(sorted(set(range(N)) - set(selected)), dtype=int)
        if available.size == 0:
            break

        X_cand_imp = imputer.transform(X[available])
        X_cand_scaled = scaler.transform(X_cand_imp)

        pick_pos, ucb, mean_eff, unc_eff = model.select_next_point(X_cand_scaled)
        next_idx = int(available[pick_pos])
        selected.append(next_idx)

        # Update training set
        X_train = np.vstack([X_train, X[[next_idx]]])
        y_train_eff = np.append(y_train_eff, y_eff[next_idx])

        # Refit imputer+scaler on expanded training set (mirrors your original "refit scaler each iter")
        imputer = SimpleImputer(strategy="mean")
        X_train_imp = imputer.fit_transform(X_train)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_imp)

        # Logging: convert predictions to true-space
        pred_true = float(-mean_eff[pick_pos]) if minimize else float(mean_eff[pick_pos])
        unc_true = float(unc_eff[pick_pos])
        obs_true = float(y_true[next_idx])

        # Stopping: when global optimum reached in true-space
        stop = (obs_true <= best_true) if minimize else (obs_true >= best_true)

        rows.append({
            "Iteration": int(it),
            "Index": next_idx,
            "Observed Target Value": obs_true,
            "Predicted Target Value": pred_true,
            "Uncertainty": unc_true,
            best_label: float(best_true),
            "Phase": "ActiveLearning",
            "Stopping Reason": f"{best_label} reached" if stop else "Continuing",
        })

        it += 1
        if stop:
            break

    df_traj = pd.DataFrame(rows)

    # Save
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)

    a_tag = alpha_tag(alpha)
    fname = f"{model_name}__{dataset_name}__a{a_tag}__seed{random_seed}__{run_kind}__{subset_key}.csv"
    fname = safe_name(fname, max_len=180)
    out_path = out_dir_path / fname

    df_traj.to_csv(out_path, index=False)
    return str(out_path), df_traj


# ============================================================
# Batch driver
# ============================================================
def run_all_warmstarts_for_dataset(
    dataset_name: str,
    original_csv_path: str,
    sampling_check_root: str = SAMPLING_CHECK_ROOT,
    out_base: str = OUT_BASE,
    model_names: Sequence[str] = MODEL_NAMES,
    alphas: Sequence[float] = ALPHAS,
):
    df_full = pd.read_csv(original_csv_path)
    y_col = get_objective_column(dataset_name)

    print("=" * 70)
    print(f"Dataset: {dataset_name}")
    print(f"Original: {original_csv_path} (N={len(df_full)})")
    print(f"Objective: {y_col} non-null={df_full[y_col].notna().sum()} unique={df_full[y_col].nunique(dropna=True)}")
    print("=" * 70)

    # Quick X sanity (using your feature config)
    X_df_raw, y_true, y_eff = build_features_and_targets(df_full, dataset_name)
    X_df_num = coerce_numeric_X(X_df_raw)
    X = X_df_num.to_numpy(dtype=float)
    print(f"[DATA CHECK] X shape={X.shape}  NaN cells={int(np.isnan(X).sum())}  NaN rows={int(np.isnan(X).any(axis=1).sum())}  NaN cols={int(np.isnan(X).any(axis=0).sum())}")

    subset_dir = os.path.join(sampling_check_root, dataset_name)
    all_csv = sorted(glob.glob(os.path.join(subset_dir, "*.csv")))
    subset_files = [p for p in all_csv if SUBSET_FILE_FILTER.match(os.path.basename(p))]

    if not subset_files:
        raise RuntimeError(f"No subset files found in {subset_dir} matching llm_experiment_*.csv")

    out_dir_warm = os.path.join(out_base, dataset_name, "ML_warmstart_from_LLMsubsets")
    out_dir_rand = os.path.join(out_base, dataset_name, "ML_random_matched_to_LLMsubsets")
    os.makedirs(out_dir_warm, exist_ok=True)
    os.makedirs(out_dir_rand, exist_ok=True)

    N = len(df_full)

    for subset_fp in subset_files:
        base = os.path.basename(subset_fp)
        seed = parse_seed_from_name(base, default=42)

        df_subset = pd.read_csv(subset_fp)
        warm_idx = recover_indices_from_subset(df_full, df_subset)
        warm_idx = np.unique(warm_idx)
        m = int(warm_idx.size)

        rng = np.random.RandomState(seed)
        rand_idx = rng.choice(np.arange(N), size=m, replace=False)

        # Short unique key for filenames
        human = safe_name(base.replace(".csv", ""), max_len=40)
        key = f"m{m}_seed{seed}_{short_hash(base, 10)}"
        subset_key = safe_name(f"{human[:25]}_{key}", max_len=70)

        print(f"\n[WARMSTART FILE] {base}")
        print(f"  seed={seed}  warmstart_points={m}  (N={N})")
        print(f"  subset_key={subset_key}")

        for model_name in model_names:
            for alpha in alphas:
                alpha = float(alpha)

                try:
                    p1, d1 = run_MLAL_warmstart(
                        df_full, dataset_name, model_name, alpha, seed, warm_idx,
                        run_kind="warm", subset_key=subset_key, out_dir=out_dir_warm
                    )
                except Exception as e:
                    print(f"  [FAIL] {model_name} alpha={alpha} warmstart: {e}")
                    continue

                try:
                    p2, d2 = run_MLAL_warmstart(
                        df_full, dataset_name, model_name, alpha, seed, rand_idx,
                        run_kind="rand", subset_key=subset_key, out_dir=out_dir_rand
                    )
                except Exception as e:
                    print(f"  [FAIL] {model_name} alpha={alpha} random: {e}")
                    continue

                al_steps_1 = int((d1["Phase"] == "ActiveLearning").sum())
                al_steps_2 = int((d2["Phase"] == "ActiveLearning").sum())

                if is_minimization_dataset(dataset_name):
                    best1 = float(np.nanmin(d1["Observed Target Value"]))
                    best2 = float(np.nanmin(d2["Observed Target Value"]))
                else:
                    best1 = float(np.nanmax(d1["Observed Target Value"]))
                    best2 = float(np.nanmax(d2["Observed Target Value"]))

                print(f"  [OK] {model_name} a={alpha}: warm(AL={al_steps_1}, best={best1:.6g}) | rand(AL={al_steps_2}, best={best2:.6g})")

    print("\nDone.")


# ============================================================
# Example usage
# ============================================================
# Run one dataset first
# run_all_warmstarts_for_dataset(
#     dataset_name="Perovskite_dataset",
#     original_csv_path=DATASET_CSVS["Perovskite_dataset"],
# )

# Run all datasets
# for ds, csvp in DATASET_CSVS.items():
#     run_all_warmstarts_for_dataset(ds, csvp)

In [5]:
# ============================================================
# Example usage
# ============================================================
# Run one dataset first
# run_all_warmstarts_for_dataset(
#     dataset_name="Perovskite_dataset",
#     original_csv_path=DATASET_CSVS["Perovskite_dataset"],
# )

# Run all datasets
for ds, csvp in DATASET_CSVS.items():
    run_all_warmstarts_for_dataset(ds, csvp)

Dataset: matbench_steels (composition)
Original: matbench_steels.csv (N=312)
Objective: yield strength non-null=312 unique=270
[DATA CHECK] X shape=(312, 14)  NaN cells=0  NaN rows=0  NaN cols=0

[WARMSTART FILE] matbench_steels_110__llm_experiment_yield_strength_trajectory_report_seed42_run3.csv
  seed=42  warmstart_points=110  (N=312)
  subset_key=matbench_steels_110__llm__m110_seed42_6a93007538
  [OK] BNN a=0.1: warm(AL=17, best=2510.3) | rand(AL=12, best=2510.3)
  [OK] BNN a=0.5: warm(AL=17, best=2510.3) | rand(AL=12, best=2510.3)
  [OK] BNN a=1.0: warm(AL=14, best=2510.3) | rand(AL=12, best=2510.3)
  [OK] XGB a=0.1: warm(AL=27, best=2510.3) | rand(AL=9, best=2510.3)
  [OK] XGB a=0.5: warm(AL=33, best=2510.3) | rand(AL=9, best=2510.3)
  [OK] XGB a=1.0: warm(AL=32, best=2510.3) | rand(AL=9, best=2510.3)
  [OK] RFR a=0.1: warm(AL=31, best=2510.3) | rand(AL=1, best=2510.3)
  [OK] RFR a=0.5: warm(AL=32, best=2510.3) | rand(AL=2, best=2510.3)
  [OK] RFR a=1.0: warm(AL=33, best=2510.3) |